 Day 3 is where we move from non-personalized popularity-based recommendations to personalized recommendations using Collaborative Filtering (User-Based).

This is the first step where user behavior drives the recommendations, not just overall popularity.


---

🚀 Day 3 — User-Based Collaborative Filtering

Goal

Recommend movies to a specific user based on the preferences of similar users.

Key concepts today:

user similarity

neighbor users

rating prediction

sparsity handling



---

🧩 Step 1 — User-Item Matrix

Collaborative filtering works on a matrix of users vs items.

import pandas as pd

# Load merged dataset from Day 1
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('../data/ml-100k/u.data', sep='\t', names=ratings_cols)

movie_cols = ['movie_id', 'title']
movies = pd.read_csv('../data/ml-100k/u.item', sep='|', header=None, encoding='latin-1', usecols=[0,1], names=movie_cols)

df = pd.merge(ratings, movies, on='movie_id')

# Create user-item matrix
user_movie_matrix = df.pivot_table(index='user_id', columns='title', values='rating')
user_movie_matrix.head()

Explanation:

Rows = users

Columns = movies

Values = ratings

Missing entries = NaN → user didn’t rate that movie


This is the core structure for collaborative filtering.


---

🧩 Step 2 — Compute User Similarity

We need similar users. Use Pearson correlation:

user_similarity = user_movie_matrix.T.corr()
user_similarity.head()

Explanation:

.T.corr() calculates correlation between users (columns become rows)

Values range: -1 to 1 → 1 = identical tastes, 0 = no correlation



---

🧩 Step 3 — Pick a User & Find Similar Users

# Example: user_id = 1
target_user = 1

similar_users = user_similarity[target_user].sort_values(ascending=False)
similar_users = similar_users.drop(target_user)  # remove self
similar_users.head(10)

Explanation:

Shows top 10 users most similar to target user

We’ll use their ratings to predict what target user may like



---

🧩 Step 4 — Predict Ratings

We calculate weighted average ratings from similar users:

import numpy as np

# Select top 5 similar users
top_sim_users = similar_users.head(5).index

# Movies rated by these similar users
similar_user_ratings = user_movie_matrix.loc[top_sim_users]

# Weighted prediction for target user
pred_ratings = similar_user_ratings.mean(axis=0)  # simple average
pred_ratings = pred_ratings[user_movie_matrix.loc[target_user].isna()]  # only unseen movies

pred_ratings = pred_ratings.sort_values(ascending=False)
pred_ratings.head(10)

Explanation:

Only consider movies the user hasn’t rated

Average ratings from similar users → predicted preference

Sorted → top 10 personalized recommendations



---

🧩 Step 5 — Example Output

Suppose user 1 hasn’t rated “Star Wars”, “Godfather”, etc.

Predicted top 10 might be:

Movie	Predicted Rating

Star Wars (1977)	4.8
Godfather (1972)	4.7
Casablanca (1942)	4.6
Schindler’s List (1993)	4.6
Titanic (1997)	4.5


Now recommendations are personalized. ✅


---

🧠 Step 6 — Why This Matters

Unlike popularity-based, now recommendations depend on user behavior

Handles cold-start partially if users have at least some ratings

Can easily extend to item-based collaborative filtering later (Day 4)



---

📌 Homework / Experiments

1. Try different target users and compare recommendations.


2. Change number of similar users (top 3, 5, 10) → see effect.


3. Experiment with Pearson vs Cosine similarity → subtle difference.


4. Find a sparsely-rated user → see how predictions get affected.




---

💡 Industry Insight:

This is essentially Netflix recommendation logic: “Users similar to you liked this.”

Early-stage collaborative filtering works surprisingly well for personalized suggestions.



---

If you want, I can also show the full code for item-based collaborative filtering, which is faster for large datasets and closer to what Netflix/Amazon actually use in production.

Do you want me to do that next?



In [1]:
# Imports

import pandas as pd

import matplotlib.pyplot as plt

import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:

rating_cols = [ 'user_id', 'movie_id', 'rating', 'timestamp' ]

ratings = pd.read_csv ( r"C:\Users\Mugundhan\ML_Learning\recommendation_system\data\ml-100k\u.data", sep = '\t', names = rating_cols )

ratings.head()

movie_cols = [ 'movie_id', 'title' ]

movies = pd.read_csv ( r"C:\Users\Mugundhan\ML_Learning\recommendation_system\data\ml-100k\u.item", sep = '|', encoding = 'latin-1', usecols = [0, 1], names = movie_cols )

movies.head()

df = pd.merge ( ratings, movies, on = 'movie_id' )

df.head()

,user_id,movie_id,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [3]:
# USER ITEM MATRIX

user_movie_matrix = df.pivot_table ( index = 'user_id', columns = 'title', values = 'rating' )

user_movie_matrix.head()

# Groups data by one or more keys (like movie title, user_id, etc.).

# Applies aggregation functions (like mean, sum, count) to other columns.

# Produces a new DataFrame with a matrix‑style layout.

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,2.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,4.0,NaN


In [4]:
# Compute User Similarity

user_similarity = user_movie_matrix.T.corr()

user_similarity.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.160841,0.11278,0.500000,0.420809,0.295410,0.258137,0.692086,-0.102062,-0.092344,...,0.061695,-0.260242,0.386346,0.029000,0.326744,5.343904e-01,0.263289,0.205616,-0.180784,0.067549
2,0.160841,1.000000,0.06742,0.148522,0.327327,0.446966,0.643675,0.585491,0.242536,0.668145,...,0.029341,-0.271163,0.214017,0.566724,0.331587,1.380822e-16,-0.011682,-0.062017,0.085960,0.479702
3,0.112780,0.067420,1.00000,-0.262600,NaN,-0.109109,0.064803,0.291937,NaN,0.311086,...,0.000000,NaN,-0.045162,0.000000,-0.137523,NaN,-0.104678,1.000000,-0.011792,NaN
4,0.500000,0.148522,-0.26260,1.000000,1.000000,-0.581318,-0.266632,0.642938,NaN,-0.301511,...,0.866025,NaN,-0.203653,NaN,0.375000,NaN,0.850992,1.000000,0.412568,NaN
5,0.420809,0.327327,NaN,1.000000,1.000000,0.241817,0.175630,0.537400,0.577350,0.087343,...,0.229532,-0.500000,0.439286,0.608581,0.484211,8.807048e-01,0.027038,0.468521,0.318163,0.346234



---

### 1️⃣ `user_movie_matrix = df.pivot_table(index='user_id', columns='title', values='rating')`
- This creates a **user–movie rating matrix**:
  - Rows = users (`user_id`)
  - Columns = movies (`title`)
  - Values = ratings
- So you end up with a table of shape **943 users × 1664 movies** (sparse, since most users haven’t rated most movies).

---

### 2️⃣ `user_similarity = user_movie_matrix.T.corr()`
- `.T` transposes the matrix → now rows = movies, columns = users.
- `.corr()` computes **pairwise correlation** between the columns (users).
- Result: a **user–user similarity matrix** based on how their ratings correlate.

---

### 3️⃣ Why the shape changes
- Your original matrix: `943 × 1664` (users × movies).
- After transpose: `1664 × 943` (movies × users).
- `.corr()` computes correlation between **users** (the columns after transpose).
- So the result is a **943 × 943 matrix** (user similarity).
- When you call `.head()`, pandas shows the first 5 rows × all 943 columns → that’s why you saw `5 rows × 943 columns`.

---

### 🔹 Intuition
- Each cell in `user_similarity` tells you how similar two users are in terms of their rating patterns.
- Values range from `-1` (opposite tastes) to `+1` (identical tastes).
- This is the basis for **user-based collaborative filtering** in recommendation systems.

---

✅ So:  
- `pivot_table` built the ratings matrix.  
- `.T.corr()` gave you a user–user similarity matrix.  
- The shape difference is because correlation is computed across users, not movies.  

Would you like me to also show you how to compute a **movie–movie similarity matrix** (so you can recommend movies similar to a given one)? That’s the other common approach.

In [5]:
# Example user_id = 1

target_user = 180

similar_users = user_similarity [ target_user ].sort_values ( ascending = False )

simiar_users = similar_users.drop ( target_user ) # remove self 

similar_users.head( 10 )

# similar_users = similar_users.drop ( target_user ) # Remove self

user_id
822    1.0
782    1.0
799    1.0
67     1.0
594    1.0
744    1.0
636    1.0
462    1.0
438    1.0
518    1.0
Name: 180, dtype: float64

In [6]:
# Predict Ratings

top_sim_users = similar_users.head(5).index # Selecting top 5 similar users

similar_user_ratings = user_movie_matrix.loc [ top_sim_users ] # Movies rated by those similar users

# print ( similar_user_ratings )

# Weighteed avg for target user

pred_ratings = similar_user_ratings.mean( axis = 0 ) # Simple avg

print ( pred_ratings )

pred_ratings = pred_ratings [ user_movie_matrix.loc [ target_user ].isna() ] # Only unseen movie

pred_ratings = pred_ratings.sort_values ( ascending = False )

pred_ratings.head(10)

title
'Til There Was You (1997)                2.0
1-900 (1994)                             NaN
101 Dalmatians (1996)                    NaN
12 Angry Men (1957)                      NaN
187 (1997)                               4.0
                                        ... 
Young Guns II (1990)                     NaN
Young Poisoner's Handbook, The (1995)    NaN
Zeus and Roxanne (1997)                  NaN
unknown                                  NaN
Á köldum klaka (Cold Fever) (1994)       NaN
Length: 1664, dtype: float64


title
To Kill a Mockingbird (1962)              5.0
Shawshank Redemption, The (1994)          5.0
Twelve Monkeys (1995)                     5.0
Beavis and Butt-head Do America (1996)    5.0
Titanic (1997)                            5.0
Live Nude Girls (1995)                    5.0
Close Shave, A (1995)                     5.0
Raiders of the Lost Ark (1981)            5.0
Vertigo (1958)                            5.0
G.I. Jane (1997)                          5.0
dtype: float64

Perfect! Let’s see Pearson vs Cosine in action for a specific user. I’ll use the MovieLens 100k dataset we’ve been working with.


---

🔹 Step 1 — Setup & User-Item Matrix

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load ratings
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('../data/ml-100k/u.data', sep='\t', names=ratings_cols)

# Load movies
movie_cols = ['movie_id', 'title']
movies = pd.read_csv('../data/ml-100k/u.item', sep='|', header=None, encoding='latin-1', usecols=[0,1], names=movie_cols)

# Merge
df = pd.merge(ratings, movies, on='movie_id')

# User-Item matrix
user_movie_matrix = df.pivot_table(index='user_id', columns='title', values='rating')
user_movie_matrix.head()


---

🔹 Step 2 — Pearson Correlation

# Choose a target user
target_user = 1

# Pearson similarity: correlation of target user with all other users
pearson_sim = user_movie_matrix.T.corrwith(user_movie_matrix.loc[target_user])
pearson_sim = pearson_sim.drop(target_user)  # remove self
pearson_sim = pearson_sim.sort_values(ascending=False)

print("Top 5 users by Pearson similarity to User 1:")
print(pearson_sim.head(5))

Explanation:

Uses correlation → accounts for personal rating scale differences

Returns users with most similar taste patterns



---

🔹 Step 3 — Cosine Similarity

# Fill NaN with 0 for cosine computation (common approach)
user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Compute cosine similarity matrix
cos_sim_matrix = cosine_similarity(user_movie_matrix_filled)
cos_sim_df = pd.DataFrame(cos_sim_matrix, index=user_movie_matrix.index, columns=user_movie_matrix.index)

# Get top 5 similar users
cos_sim_users = cos_sim_df[target_user].drop(target_user).sort_values(ascending=False)

print("\nTop 5 users by Cosine similarity to User 1:")
print(cos_sim_users.head(5))

Explanation:

Cosine uses the angle between rating vectors

Sensitive to absolute ratings (high/low rating bias)



---

🔹 Step 4 — Compare Pearson vs Cosine

You’ll notice:

Pearson may give high similarity to users who rate differently in scale but same taste.

Cosine may rank some of them lower if their ratings are consistently higher or lower.


Example Output (illustrative):

Top 5 users by Pearson similarity:
User 123: 0.92
User 87: 0.90
User 45: 0.89
User 78: 0.88
User 56: 0.87

Top 5 users by Cosine similarity:
User 87: 0.95
User 45: 0.94
User 56: 0.92
User 123: 0.90
User 78: 0.89

Notice subtle differences in ranking.


---

🧠 Key Takeaways

1. Pearson correlation → focuses on patterns, ignores rating scale, better for explicit ratings.


2. Cosine similarity → sensitive to absolute scale, better for implicit feedback (clicks, views).


3. Choice of similarity metric affects which users are chosen as neighbors, which in turn affects personalized recommendations.




---


In [7]:
# Load ratings
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('../data/ml-100k/u.data', sep='\t', names=ratings_cols)

# Load movies
movie_cols = ['movie_id', 'title']
movies = pd.read_csv('../data/ml-100k/u.item', sep='|', header=None, encoding='latin-1', usecols=[0,1], names=movie_cols)

# Merge
df = pd.merge(ratings, movies, on='movie_id')

# User-Item matrix
user_movie_matrix = df.pivot_table(index='user_id', columns='title', values='rating')
user_movie_matrix.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,2.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,4.0,NaN


In [8]:
# Pearson Cor relation

target_user = 1

# Pearson similarity: correlation of target user with all other users
pearson_sim = user_movie_matrix.T.corrwith(user_movie_matrix.loc[target_user])
pearson_sim = pearson_sim.drop(target_user)  # remove self
pearson_sim = pearson_sim.sort_values(ascending=False)

print("Top 5 users by Pearson similarity to User 1:")
print(pearson_sim.head(5))


Top 5 users by Pearson similarity to User 1:
user_id
351    1.0
39     1.0
531    1.0
866    1.0
273    1.0
dtype: float64


C:\Users\Mugundhan\ML_Learning\recommendation_system\venv_ml\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
C:\Users\Mugundhan\ML_Learning\recommendation_system\venv_ml\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\Mugundhan\ML_Learning\recommendation_system\venv_ml\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
C:\Users\Mugundhan\ML_Learning\recommendation_system\venv_ml\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\Mugundhan\ML_Learning\recommendation_system\venv_ml\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [9]:
# Cosine Similarity

# Fill NaN with 0 for cosine computation (common approach)
user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Compute cosine similarity matrix
cos_sim_matrix = cosine_similarity(user_movie_matrix_filled)
cos_sim_df = pd.DataFrame(cos_sim_matrix, index=user_movie_matrix.index, columns=user_movie_matrix.index)

# Get top 5 similar users
cos_sim_users = cos_sim_df[target_user].drop(target_user).sort_values(ascending=False)

print("\nTop 5 users by Cosine similarity to User 1:")
print(cos_sim_users.head(5))



Top 5 users by Cosine similarity to User 1:
user_id
916    0.566293
864    0.549277
457    0.541364
92     0.539415
268    0.538883
Name: 1, dtype: float64


# Explaination

Perfect. This is actually the correct point to slow down and rebuild intuition.

Day 3 is the FIRST time recommendation systems become:

relationship-based ML

So confusion here is normal.

Let’s rebuild the whole thing VERY simply.


---

🎯 What Are We ACTUALLY Trying To Do?

Suppose:

You like:
- Interstellar
- Inception
- Batman

Now system asks:

"Who else likes similar movies?"

Suppose User 25 also likes:

Interstellar

Inception

Batman


AND additionally likes:

Tenet


Then system says:

"You may also like Tenet"

THAT is collaborative filtering.


---

🧠 CORE IDEA OF DAY 3

We are NOT learning from movie features.

We are learning from:

similar people

That’s all.


---

🔥 STEP 1 — User-Movie Matrix

Suppose we have:

User	Batman	Titanic	Interstellar

User1	5	1	5
User2	4	1	5
User3	1	5	2


Meaning:

User1 likes sci-fi

User2 likes sci-fi

User3 likes romance


Immediately we can see:

User1 ≈ User2


---

❓WHY CREATE MATRIX?

Because ML/math needs numbers arranged properly.

Rows:

users

Columns:

movies

Values:

ratings

This structure allows us to compare users mathematically.


---

🧠 What Does Similarity Mean?

We need to answer:

"How similar are User1 and User2?"

Similarity score:

high → similar taste

low → different taste



---

🔥 Pearson Similarity (Simple Meaning)

Pearson asks:

"Do both users LIKE and DISLIKE same movies?"

NOT exact ratings.

Pattern matters.


---

Example

Movie	User1	User2

Batman	5	4
Titanic	1	2
Interstellar	5	4


Pattern SAME:

both like sci-fi

both dislike romance


So Pearson says:

VERY SIMILAR

even though exact ratings differ slightly.


---

🔥 Cosine Similarity (Simple Meaning)

Cosine asks:

"Are rating vectors pointing in same direction?"

Less intuitive initially.

Simple understanding:

It compares actual rating values more directly.


---

IMPORTANT DIFFERENCE

Suppose:

Movie	User1	User2

Batman	5	3
Titanic	1	1
Interstellar	5	3


Pearson says:

Same taste pattern ✅

Cosine says:

Ratings differ somewhat ⚠️

Because:

User2 generally gives lower ratings.



---

🧠 SIMPLE MEMORY TRICK

Pearson:

"Do they LIKE same things?"

Cosine:

"Are their rating numbers close?"


---

🚀 NOW THE IMPLEMENTATION FLOW


---

STEP 1 — Create Matrix

user_movie_matrix = df.pivot_table(
    index='user_id',
    columns='title',
    values='rating'
)

This creates:

user	movie1	movie2

1	5	NaN
2	4	3



---

STEP 2 — Pearson Similarity

user_similarity = user_movie_matrix.T.corr()

This compares:

user vs user

Output:

	User1	User2	User3

User1	1	0.9	0.1
User2	0.9	1	0.2


Meaning:

User1 & User2 very similar.



---

❓WHY .T ?

Because: Initially matrix is:

rows	columns

users	movies


But correlation compares columns.

So transpose (.T) converts:

rows	columns

movies	users


Now correlation happens between users.


---

STEP 3 — Find Similar Users

target_user = 1

similar_users = user_similarity[target_user]

Meaning:

Find users similar to User1


---

STEP 4 — Sort Similar Users

similar_users.sort_values(ascending=False)

Now we get:

User	Similarity

25	0.95
12	0.90


Meaning:

these users have similar taste.



---

STEP 5 — Recommendation Logic

Now system thinks:

"Movies liked by similar users
but NOT watched by target user"

That becomes recommendation.


---

🎯 ENTIRE DAY 3 IN ONE SENTENCE

Collaborative filtering means:

Find users with similar taste,
then recommend what they liked.

That is ALL.


---

⚠️ WHY DAY 3 FEELS CONFUSING

Because this is where ML changes from:

feature-based learning

to:

relationship-based learning

This is the major conceptual shift.


---

🔥 MOST IMPORTANT THING

DO NOT memorize formulas now.

First understand:

WHY similarity needed

WHY matrix created

WHY users compared

WHY recommendations come from neighbors


Once intuition becomes clear, the formulas become easy.

